In [1]:
import polars as pl
from scipy.stats import norm
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)
import plotly.express as px

In [2]:
budget = (
    pl.concat(
        [
            pl.read_json("budget2024.json", infer_schema_length=None),
            pl.read_json("budget2025.json", infer_schema_length=None),
        ]
    )
    .unnest("data")
    .unnest("budgetVsActualV2")
)

In [3]:
NON_RECURRING_REPAIRS = "Non-Recurring-  Repairs"
monthly_expenses = (
    budget["expenses"]
    .explode()
    .struct.unnest()
    .select("category", "monthlyBreakdown")
    .explode("monthlyBreakdown")
    .unnest("monthlyBreakdown")
    .with_columns(
        pl.col("category").cast(
            pl.Enum(
                [
                    "Utilities",
                    "Payroll Expenses",
                    "Compliance & Monitoring",
                    "Insurance",
                    NON_RECURRING_REPAIRS,
                    "Building Operating Expenses",
                    "Taxes",
                    "Reserve Contributions/Transfers",
                    "Administrative Expenses",
                ]
            )
        )
    )
)

In [4]:
def extract_cashflow(cashflow: pl.Series):
    return (
        cashflow.explode()
        .struct.unnest()
        .select("subAccounts", headline_category="category")
        .explode("subAccounts")
        .unnest("subAccounts")
        .select(
            "headline_category",
            "category",
            pl.col("annual").struct.field("transactions"),
        )
        .explode("transactions")
        .unnest("transactions")
        .with_columns(pl.col("date").str.to_date())
        .drop(["transactionType", "__typename"])
        .drop_nulls()
    )


cashflow = pl.concat(
    [
        extract_cashflow(budget["incomes"]).with_columns(is_income=pl.lit(True)),
        extract_cashflow(budget["expenses"]).with_columns(is_income=pl.lit(False)),
    ]
)
cashflow

headline_category,category,id,description,amount,date,is_income
str,str,i64,str,f64,date,bool
"""Other Income""","""Interest Income""",48558424,"""Recording of Opening balance f…",6.82,2024-01-31,true
"""Other Income""","""Interest Income""",47587987,"""Recording the Opening balance …",6.24,2024-02-29,true
"""Other Income""","""Insurance Recovery Proceeds""",42605392,"""Brownstone Agency Inc - Water …",20065.98,2024-07-16,true
"""Other Income""","""Working Capital Contribution f…",43793393,"""8/2024 Payment from 07/24/2024…",1181.38,2024-08-22,true
"""Other Income""","""Working Capital Contribution f…",44077769,"""REVERSED - Manually""",-1181.38,2024-08-27,true
…,…,…,…,…,…,…
"""Utilities""","""Water / Sewer""",53558125,"""Water & Wastewater Bill – Date…",845.09,2025-06-20,false
"""Utilities""","""Water / Sewer""",54542432,"""""",819.86,2025-07-22,false
"""Utilities""","""Water / Sewer""",55471336,"""Water & Wastewater Bill – Date…",857.39,2025-08-21,false


In [5]:
MONTHS_TO_SIMULATE = 13
SIMULATIONS = 10000
STARTING_BALANCE = 15711.40
BUDGET_INCREASES = (budget_increase / 100.0 + 0.05 for budget_increase in range(-2, 3))
cashflow_by_month_by_increase = [
    (
        budget_increase,
        cashflow.with_columns(
            pl.col("amount")
            * pl.when(pl.col("is_income")).then(1 + budget_increase).otherwise(-1)
        )
        .group_by(
            year=pl.col("date").dt.year(),
            month=pl.col("date").dt.month(),
        )
        .agg(pl.col("amount").sum()),
    )
    for budget_increase in BUDGET_INCREASES
]

simulations = pl.concat(
    pl.collect_all(
        (
            cashflow_by_month.lazy()
            .select(
                (
                    pl.col("amount").sample(MONTHS_TO_SIMULATE, shuffle=True).cum_sum()
                    + pl.lit(STARTING_BALANCE)
                ).min(),
            )
            .with_columns(
                simulation_id=pl.lit(simulation_id),
                budget_increase=pl.lit(budget_increase),
            )
            for (
                budget_increase,
                cashflow_by_month,
            ) in cashflow_by_month_by_increase
            for simulation_id in range(0, SIMULATIONS)
        )
    )
)

In [6]:
fig = px.pie(
    simulations.group_by(
        "budget_increase",
        ruinous=pl.col("amount") < 0,
    )
    .len()
    .with_columns(
        ruinous=pl.when("ruinous").then(pl.lit("Ruinous")).otherwise(pl.lit("Safe"))
    )
    .sort("budget_increase"),
    names="ruinous",
    values="len",
    facet_col="budget_increase",
)
fig.show(renderer="notebook_connected")

In [7]:
simulations.group_by("budget_increase").agg(
    pl.col("amount").mean() + pl.col("amount").std() * norm.ppf(0.95)
)

budget_increase,amount
f64,f64
0.03,24970.405535
0.04,25472.498945
0.05,25872.304679
0.06,26282.666427
0.07,26486.230349


In [8]:
fig = px.bar(
    extract_cashflow(budget["expenses"]).with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="category",
)
fig.show(renderer="notebook_connected")

In [9]:
nrr_actual = (
    monthly_expenses.filter(pl.col("category") != NON_RECURRING_REPAIRS)
    .group_by(["month", "year"])
    .agg(pl.col("actual").sum())["actual"]
)
projected_monthly_budget = nrr_actual.mean() + nrr_actual.std() * norm.ppf(0.95)
CURRENT_MONTHLY_BUDGET = 19797.62
projected_increase = (
    projected_monthly_budget - CURRENT_MONTHLY_BUDGET
) / CURRENT_MONTHLY_BUDGET

In [10]:
projected_monthly_budget

np.float64(33004.77886008375)

In [11]:
projected_increase

np.float64(0.6671084130356959)